# GPT sur Tiny Shakespeare



In [1]:
from pathlib import Path
import math
import random

import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments

# ====================== CONFIGURATION ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ====================== CHEMINS ======================
DATA_PATH = Path("./data/tiny_shakespeare.txt")
OUTPUT_DIR = Path("./outputs/tiny_shakespeare_gpt")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ====================== PARAMÈTRES ======================
MODEL_NAME = "gpt2"
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
SEARCH_EPOCHS = 3
FINAL_EPOCHS = 3
N_TRIALS = 2

N_LAYER = 4
N_HEAD = 4
N_EMBD = 256

print("CUDA available:", torch.cuda.is_available())
print("Data path:", DATA_PATH)
print("Output dir:", OUTPUT_DIR)
print("Model:", MODEL_NAME)

CUDA available: False
Data path: data\tiny_shakespeare.txt
Output dir: outputs\tiny_shakespeare_gpt
Model: gpt2


In [ ]:
%pip install --quiet torch transformers datasets tokenizers accelerate matplotlib optuna

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing Tiny Shakespeare file: {DATA_PATH}")

with DATA_PATH.open("r", encoding="utf-8") as handle:
    lines = [line.strip() for line in handle.readlines()]

lines = [line for line in lines if line]
print("Total non-empty lines:", len(lines))
print("Sample line:", lines[0])

raw_dataset = Dataset.from_dict({"text": lines})
split_80_20 = raw_dataset.train_test_split(test_size=0.20, seed=SEED)
train_raw = split_80_20["train"]
temp_raw = split_80_20["test"]
split_10_10 = temp_raw.train_test_split(test_size=0.50, seed=SEED)
val_raw = split_10_10["train"]
test_raw = split_10_10["test"]

print("Train rows:", len(train_raw))
print("Validation rows:", len(val_raw))
print("Test rows:", len(test_raw))

Total non-empty lines: 32777
Sample line: First Citizen:
Train rows: 26221
Validation rows: 3278
Test rows: 3278


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

train_tokenized = train_raw.map(
    lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)
val_tokenized = val_raw.map(
    lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)
test_tokenized = test_raw.map(
    lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)

print(train_tokenized[0])
print("Tokenizer vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\afagn\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/26221 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

{'input_ids': [7279, 2875, 11, 9961, 11, 3252, 290, 4517, 3541, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [4]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=MAX_LENGTH,
    n_ctx=MAX_LENGTH,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


def model_init():
    return GPT2LMHeadModel(config)


training_args_search = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "optuna"),
    num_train_epochs=SEARCH_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

search_trainer = Trainer(
    model_init=model_init,
    args=training_args_search,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)


def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }


print("Ready for Optuna search")

Ready for Optuna search


In [ ]:
best_run = search_trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=N_TRIALS,
)

print("Best run:")
print(best_run)

best_learning_rate = best_run.hyperparameters["learning_rate"]
best_weight_decay = best_run.hyperparameters["weight_decay"]

final_training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "final"),
    num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=best_learning_rate,
    weight_decay=best_weight_decay,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

model = model_init()
trainer = Trainer(
    model=model,
    args=final_training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

train_result = trainer.train()
val_results = trainer.evaluate()
test_results = trainer.evaluate(test_tokenized, metric_key_prefix="test")

val_loss = val_results["eval_loss"]
test_loss = test_results["test_loss"]
val_perplexity = math.exp(val_loss)
test_perplexity = math.exp(test_loss)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation perplexity: {val_perplexity:.2f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test perplexity: {test_perplexity:.2f}")

In [ ]:
model = trainer.model
model.to(final_training_args.device)
model.eval()

prompts = [
    "To be, or not to be",
    "The king said",
    "Tomorrow and tomorrow",
]

for prompt in prompts:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(final_training_args.device)
    generated_ids = model.generate(
        input_ids,
        max_new_tokens=80,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
    print("\nPROMPT:", prompt)
    print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))